# Route E: redox across the mitotic-to-endocycle switch in the fly ovary

Follicle cells divide until stage 6, then endocycle. Both are the same cell type in the same tissue, so this is a cleaner test than the salivary gland. Pick the dataset in Cell 1, then **Runtime → Run all**. Data download from GEO into *Colab Notebooks/route_e_ovary*.

In [ ]:
#@title Cell 1 - Settings and connect Google Drive
import gzip, io, os, re, subprocess, tarfile, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print

#@markdown **GSE146040**: adult ovary single cells, Jevitt et al. 2020 (Deng lab), ~35,000 cells.
#@markdown **GSE136162**: adult ovary single cells, Rust et al. 2020 (Nystul lab).
#@markdown Both hold mitotic follicle cells (stages 1-6) and endocycling follicle cells (stages 7-10) from the same tissue.
GSE = "GSE146040"  #@param ["GSE146040", "GSE136162"]
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_e_ovary"  #@param {type:"string"}
GTF_PATH = "/content/drive/MyDrive/Colab Notebooks/route_b_fly/Drosophila_melanogaster.gtf.gz"  #@param {type:"string"}
N_CLUSTERS = 18  #@param {type:"integer"}
MIN_GENES = 400  #@param {type:"integer"}
MIN_COUNTS = 1000  #@param {type:"integer"}
N_RANDOM_SETS = 2000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}
DATA_DIR = os.environ.get("ROUTE_E_DATA", DATA_DIR)
GTF_PATH = os.environ.get("ROUTE_E_GTF", GTF_PATH)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
GDIR = os.path.join(DATA_DIR, GSE)
os.makedirs(GDIR, exist_ok=True)
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 16); pd.set_option("display.max_rows", 220)


class StopHere(Exception):
    """Stops the notebook with a plain-language message."""


print("Dataset:", GSE, "| folder:", GDIR)

In [ ]:
#@title Cell 2 - Download the single-cell files from GEO
BASE = os.environ.get("ROUTE_E_GEO", "https://ftp.ncbi.nlm.nih.gov/geo")


def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"   already here: {os.path.basename(dest)}"); return True
    r = subprocess.run(["curl", "-fsSL", "-m", "3600", "-o", dest + ".part", url], capture_output=True, text=True)
    if r.returncode != 0:
        if os.path.exists(dest + ".part"): os.remove(dest + ".part")
        return False
    os.replace(dest + ".part", dest)
    print(f"   downloaded {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.1f} MB)"); return True


def listing(url, tries=3):
    for _ in range(tries):
        r = subprocess.run(["curl", "-fsSL", "-m", "180", url], capture_output=True, text=True)
        if r.returncode == 0:
            return [h for h in re.findall(r'href="([^"?/][^"]*)"', r.stdout) if not h.startswith("http")]
    return []


def series_dir(acc):
    return f"{BASE}/series/{acc[:-3]}nnn/{acc}/"


def data_files(acc):
    """Supplementary files of a series: at series level, under its sub-series, or per sample."""
    sdir = series_dir(acc) + "suppl/"
    names = [h for h in listing(sdir) if not h.endswith("/") and h != "filelist.txt"]
    if names:
        return [(sdir + n, n) for n in names]
    out = []
    mdir = series_dir(acc) + "matrix/"
    for mn in [h for h in listing(mdir) if h.endswith("_series_matrix.txt.gz")]:
        mp = os.path.join(GDIR, mn)
        if not fetch(mdir + mn, mp):
            continue
        with gzip.open(mp, "rt", errors="replace") as h:
            head = h.read(400000)
        subs = re.findall(r"SuperSeries of: (GSE\d+)", head)
        for sub in dict.fromkeys(subs):                       # a SuperSeries keeps its files in the sub-series
            print(f"   {acc} is a SuperSeries; looking inside {sub}")
            out += data_files(sub)
        if not subs:
            gsms = dict.fromkeys(re.findall(r"GSM\d+", head))
            print(f"   no files at series level; checking {len(gsms)} samples one by one")
            for g in gsms:
                gd = f"{BASE}/samples/{g[:-3]}nnn/{g}/suppl/"
                out += [(gd + n, n) for n in listing(gd, tries=1) if not n.endswith("/") and n != "filelist.txt"]
    return out


EBI = os.environ.get("ROUTE_E_EBI", "https://ftp.ebi.ac.uk/pub/databases/microarray/data/atlas/sc_experiments")


def ebi_files(acc):
    """Some studies put only raw sequencing on GEO. EBI's Single Cell Expression Atlas holds the ready-made matrix."""
    e = "E-GEOD-" + re.sub(r"\D", "", acc)
    d = f"{EBI}/{e}/"
    names = [n for n in listing(d) if re.search(r"\.(mtx|mtx_rows|mtx_cols|tsv)$", n)]
    keep = [n for n in names if ".mtx" in n or "cell_metadata" in n or n.startswith("ExpDesign")]
    if keep:
        print(f"   GEO has no processed files; using EBI Single Cell Expression Atlas entry {e}")
    return [(d + n, n) for n in keep]


found = data_files(GSE) or ebi_files(GSE)
print(f"Files to download: {len(found)}")
for _, n in found[:12]:
    print("   ", n)
got = [os.path.join(GDIR, n) for u, n in found if fetch(u, os.path.join(GDIR, n))]
if not got:
    got = [os.path.join(GDIR, n) for n in sorted(os.listdir(GDIR))
           if not n.endswith((".part", ".csv", ".png")) and "series_matrix" not in n]
    if got:
        print("GEO did not answer; using files already in your folder:", [os.path.basename(g) for g in got])
if not got:
    raise StopHere(f"No data files found for {GSE} at the series, sub-series or sample level. "
                   "GEO may be busy: try running this cell again. If it keeps failing, send me this message.")
for p in [g for g in got if g.lower().endswith(".tar")]:        # unpack per-sample bundles
    with tarfile.open(p) as tf:
        tf.extractall(os.path.join(GDIR, "unpacked"))
    print(f"   unpacked {os.path.basename(p)}")
files = []
for root, _, names in os.walk(GDIR):
    files += [os.path.join(root, n) for n in names
              if not n.endswith((".part", ".tar")) and "series_matrix" not in n]
print(f"\n{len(files)} data files available.")

In [ ]:
#@title Cell 3 - Load the count matrices (cells x genes)
def read_mtx_set(mtx, bar, feat):
    """A 10x triplet: matrix.mtx(.gz), barcodes.tsv(.gz), features/genes.tsv(.gz)."""
    from scipy.io import mmread
    op = gzip.open if mtx.endswith(".gz") else open
    with op(mtx, "rb") as h:
        M = sp.csr_matrix(mmread(h))
    rd = lambda p: [l.rstrip("\n").split("\t") for l in (gzip.open(p, "rt") if p.endswith(".gz") else open(p))]
    cells_ = [r[0] for r in rd(bar)]
    rows = rd(feat)
    names = [r[1] if len(r) > 1 else r[0] for r in rows]           # column 2 is the gene symbol
    if M.shape[0] == len(names) and M.shape[1] == len(cells_):
        M = M.T.tocsr()
    return M, cells_, names


def read_dense(path):
    op = gzip.open(path, "rt") if path.endswith(".gz") else open(path)
    first = op.readline(); op.close()
    sep = "," if first.count(",") > first.count("\t") else "\t"
    d = pd.read_csv(path, sep=sep, index_col=0, low_memory=False)
    d = d.apply(pd.to_numeric, errors="coerce").dropna(how="all")
    if d.shape[0] < d.shape[1]:                                     # genes should be the long side
        d = d.T
    return sp.csr_matrix(d.to_numpy(dtype=np.float32).T), list(d.columns), list(d.index)


def to_symbols(gs):
    """EBI files name genes by FlyBase ID; turn them into gene names using the annotation."""
    if np.mean([str(g).startswith("FBgn") for g in gs[:500]]) <= 0.5:
        return list(gs), False
    if not os.path.exists(GTF_PATH):
        raise StopHere(f"The matrix uses FlyBase IDs and the annotation is not at {GTF_PATH}. It is the same file "
                       "used in Route B; fix GTF_PATH in Cell 1.")
    global _ID2SYM
    if "_ID2SYM" not in globals():
        _ID2SYM = {}
        with gzip.open(GTF_PATH, "rt") as h:
            for line in h:
                if "\tgene\t" not in line:
                    continue
                a = re.search(r'gene_id "([^"]+)"', line); b = re.search(r'gene_name "([^"]+)"', line)
                if a and b:
                    _ID2SYM[a.group(1)] = b.group(1)
    return [_ID2SYM.get(re.sub(r"\.\d+$", "", str(g)), str(g)) for g in gs], True


def rank_of(name):
    """Lower is better. GEO per-sample files keep the library structure, so they come first.
    The EBI entry ships the same data three times over; take the filtered raw counts."""
    n = name.lower()
    if "aggregated" in n or n.startswith("e-geod") or n.startswith("e-mtab"):
        if "filtered_normalised" in n:
            return 3
        return 1 if "filtered" in n else 2
    return 0


found_blocks = []
by_dir = {}
for p in files:
    by_dir.setdefault(os.path.dirname(p), []).append(p)
for d, ps in sorted(by_dir.items()):
    triplets = {}
    for p in ps:
        n = os.path.basename(p).lower()
        key = re.sub(r"((barcodes|features|genes|matrix)\.(tsv|mtx)|\.mtx(_rows|_cols)?)(\.gz)?$", "", n)
        kind = ("mtx" if n.endswith((".mtx", ".mtx.gz")) or ("matrix" in n and ".mtx" in n)
                else "bar" if "barcode" in n or n.endswith((".mtx_cols", ".mtx_cols.gz"))
                else "feat" if "feature" in n or "genes" in n or n.endswith((".mtx_rows", ".mtx_rows.gz"))
                else None)
        if kind:
            triplets.setdefault(key, {})[kind] = p
    for key, t in sorted(triplets.items()):
        if {"mtx", "bar", "feat"} <= set(t):
            M, cs, gs = read_mtx_set(t["mtx"], t["bar"], t["feat"])
            lib = (key.strip("_-. ") or os.path.basename(d)) or "library1"
            found_blocks.append({"rank": rank_of(lib), "lib": lib, "M": M, "genes": gs, "cells": cs})
if not found_blocks:                                                # no matrix triplets: try a dense table
    for p in sorted(files, key=os.path.getsize, reverse=True):
        if re.search(r"\.(csv|tsv|txt|tab)(\.gz)?$", p, re.I):
            try:
                M, cs, gs = read_dense(p)
            except Exception as ex:
                print(f"   {os.path.basename(p)}: could not read ({ex})"); continue
            if M.shape[0] > 200 and M.shape[1] > 2000:
                lib = re.sub(r"\.(csv|tsv|txt|tab)(\.gz)?$", "", os.path.basename(p), flags=re.I)
                found_blocks.append({"rank": rank_of(lib), "lib": lib, "M": M, "genes": gs, "cells": cs})
if not found_blocks:
    print("Files found:", [os.path.basename(f) for f in files][:40])
    raise StopHere("None of the files could be read as a single-cell matrix. Send me this output.")

print("Matrices found in your folder:")
for b in sorted(found_blocks, key=lambda b: b["rank"]):
    print(f"   {b['lib']}: {b['M'].shape[0]:,} cells x {b['M'].shape[1]:,} genes")
best = min(b["rank"] for b in found_blocks)
chosen = [b for b in found_blocks if b["rank"] == best]
if best > 0 and len(chosen) > 1:                                    # EBI ships one dataset in several forms
    chosen = chosen[:1]
if len(chosen) < len(found_blocks):
    print(f"\nSeveral versions of this dataset are present. Using "
          f"{'the per-sample files' if best == 0 else chosen[0]['lib']} and ignoring the rest, so that one dataset "
          "is not counted twice.")
blocks, groups = [], []
for b in chosen:
    gs, converted = to_symbols(b["genes"])
    blocks.append((b["M"], gs))
    groups.append(pd.DataFrame({"library": b["lib"], "barcode": b["cells"]}))
    print(f"Using {b['lib']}: {b['M'].shape[0]:,} cells x {b['M'].shape[1]:,} genes"
          + (" (FlyBase IDs converted to gene names)" if converted else ""))

shared = sorted(set.intersection(*[set(g) for _, g in blocks]))
if len(shared) < 3000:
    raise StopHere(f"Only {len(shared)} gene names are shared across the libraries. Send me this message.")
X = sp.vstack([M[:, [{g: i for i, g in enumerate(gs)}[s] for s in shared]] for M, gs in blocks]).tocsr()
cells = pd.concat(groups, ignore_index=True)
genes = pd.Index(shared)
X.data = np.nan_to_num(X.data)
cells["n_counts"] = np.asarray(X.sum(axis=1)).ravel()
cells["n_genes"] = np.asarray((X > 0).sum(axis=1)).ravel()
mito = [i for i, g in enumerate(genes) if g.startswith(("mt:", "mt-"))]
cells["pct_mito"] = 100 * np.asarray(X[:, mito].sum(axis=1)).ravel() / np.maximum(cells.n_counts, 1) if mito else 0.0
keep = (cells.n_genes >= MIN_GENES) & (cells.n_counts >= MIN_COUNTS) & (cells.pct_mito < 30)
print(f"\nCells before filtering: {len(cells):,}  |  kept: {int(keep.sum()):,}")
cells, X = cells[keep.to_numpy()].reset_index(drop=True), X[np.flatnonzero(keep.to_numpy())]
detected = np.asarray((X > 0).sum(axis=0)).ravel()
gkeep = detected >= 10
X, genes = X[:, np.flatnonzero(gkeep)].tocsr(), genes[gkeep]
print(f"Genes kept (detected in >=10 cells): {len(genes):,}  |  libraries: {cells.library.nunique()}")
display(cells.groupby("library").agg(cells=("barcode", "size"), median_counts=("n_counts", "median")))

In [ ]:
#@title Cell 4 - Normalise, cluster, and name the clusters from marker genes
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

gpos = {g: i for i, g in enumerate(genes)}
cpm = X.multiply(1e4 / np.maximum(cells.n_counts.to_numpy()[:, None], 1)).tocsr()
Ln = cpm.copy(); Ln.data = np.log1p(Ln.data)                       # log normalised expression per cell
v = np.asarray(Ln.power(2).mean(axis=0)).ravel() - np.asarray(Ln.mean(axis=0)).ravel() ** 2
hv = np.argsort(v)[-2000:]
Z = Ln[:, hv].toarray()
Z = (Z - Z.mean(0)) / np.where(Z.std(0) > 0, Z.std(0), 1)
svd = TruncatedSVD(n_components=30, random_state=SEED).fit_transform(np.clip(Z, -10, 10))
svd = svd / np.maximum(np.linalg.norm(svd, axis=1, keepdims=True), 1e-9)
cells["cluster"] = KMeans(n_clusters=N_CLUSTERS, n_init=5, random_state=SEED).fit_predict(svd).astype(str)


def score(gs):
    """Mean normalised expression of a gene set, per cell."""
    idx = [gpos[g] for g in gs if g in gpos]
    return np.asarray(Ln[:, idx].mean(axis=1)).ravel() if idx else np.zeros(len(cells)), len(idx)


MARKERS = {
    "follicle cell (somatic epithelium)": "eya br Fas3 CadN Fas2 Sox14 mirr pnt",
    "germline": "vas nos bam orb osk Rbp9 corolla",
    "escort / inner germarial sheath": "ptc hh tj Wnt4 bnl",
    "muscle sheath": "Mhc Mlc1 up zip Act57B",
    "stalk and polar cells": "upd1 LamC cas Fas3",
    "mitotic (cycling)": "stg CycB polo aurB Det pav PCNA CycA Incenp",
    "early follicle (stages 1-6)": "CadN Fas2 ct SPARC",
    "mid follicle (stages 7-10, endocycling)": "br mirr mid Yp1 Yp2 Yp3",
    "late follicle (stages 9+, endocycling)": "Sox14 Vha16-1 Pvf1 dpp cv-2",
    "vitelline membrane / chorion (amplifying)": "Vm26Aa Vm26Ab Vm32E Vm34Ca Cp36 Cp38 Cp15 Cp16 dec-1 psd",
}
tab = {}
for name, gs in MARKERS.items():
    s, n = score(gs.split())
    tab[f"{name} ({n})"] = pd.Series(s).groupby(cells.cluster.to_numpy()).mean()
tab = pd.DataFrame(tab)
zt = (tab - tab.mean()) / tab.std().replace(0, 1)
zt["cells"] = cells.cluster.value_counts()
display(zt.round(2).sort_values(zt.columns[0], ascending=False))
print("Columns are z-scores across clusters: a high value means that cluster expresses those markers most.")

In [ ]:
#@title Cell 5 - Pick mitotic and endocycling follicle cells, and check them
fc_col = [c for c in zt.columns if c.startswith("follicle cell")][0]
gl_col = [c for c in zt.columns if c.startswith("germline")][0]
mu_col = [c for c in zt.columns if c.startswith("muscle")][0]
mit_col = [c for c in zt.columns if c.startswith("mitotic")][0]
early_col = [c for c in zt.columns if c.startswith("early follicle")][0]
mid_col = [c for c in zt.columns if c.startswith("mid follicle")][0]
late_col = [c for c in zt.columns if c.startswith("late follicle")][0]
amp_col = [c for c in zt.columns if c.startswith("vitelline")][0]

esc_col = [c for c in zt.columns if c.startswith("escort")][0]
# Follicle cells are defined by EXCLUDING the other lineages. The general epithelial markers (CadN, Fas2, br)
# fade as follicle cells mature, so requiring them would throw away exactly the late, most polyploid cells.
is_fc = (zt[gl_col] < 0.5) & (zt[mu_col] < 0.5) & (zt[esc_col] < 0.5)
stage_raw = zt[[mid_col, late_col, amp_col]].mean(axis=1) - zt[early_col]
# Rank the follicle clusters among themselves, so the split is early vs late WITHIN the lineage
f = zt.index[is_fc]
stage = (stage_raw - stage_raw[f].mean()) / (stage_raw[f].std() or 1)
mitr = (zt[mit_col] - zt.loc[f, mit_col].mean()) / (zt.loc[f, mit_col].std() or 1)
role = {}
for cl in zt.index:
    if not is_fc[cl]:
        role[cl] = "other cell type"
    elif mitr[cl] > 0 and stage[cl] < 0:
        role[cl] = "mitotic follicle cell"
    elif mitr[cl] < 0 and stage[cl] > 0:
        role[cl] = "endocycling follicle cell"
    else:
        role[cl] = "follicle cell, unclear stage"
cells["role"] = cells.cluster.map(role)
summary = pd.DataFrame({"role": pd.Series(role), "mitotic_z": mitr.round(2),
                        "stage_score": stage.round(2), "cells": cells.cluster.value_counts()})
display(summary.sort_values(["role", "stage_score"]))
print(cells.role.value_counts().to_string())
MIT, ENDO = "mitotic follicle cell", "endocycling follicle cell"
if not {MIT, ENDO} <= set(cells.role.unique()):
    raise StopHere("Could not find both mitotic and endocycling follicle cell clusters. Send me the tables above; "
                   "raising N_CLUSTERS in Cell 1 often separates them.")

# Checks: the two groups must differ in the expected way
for name, gs in [("mitotic genes (expected much lower in endocycling cells)", MARKERS["mitotic (cycling)"]),
                 ("early markers CadN, Fas2, ct (expected lower in endocycling cells)", MARKERS["early follicle (stages 1-6)"]),
                 ("yolk and mid markers (expected higher in endocycling cells)", MARKERS["mid follicle (stages 7-10, endocycling)"])]:
    s, n = score(gs.split())
    a, b = s[(cells.role == MIT).to_numpy()].mean(), s[(cells.role == ENDO).to_numpy()].mean()
    print(f"{name}: mitotic {a:.2f}, endocycling {b:.2f}, difference {b - a:+.2f}")
print(f"\nMedian counts per cell: mitotic {cells.n_counts[cells.role == MIT].median():,.0f}, "
      f"endocycling {cells.n_counts[cells.role == ENDO].median():,.0f}")

# Dissociation stress: pulling a tissue apart switches on heat shock and immediate-early genes, and it hits
# large fragile cells hardest. If one group is more stressed, that alone shifts many genes.
DISSOC = "Hsp70Aa Hsp70Ab Hsp68 Hsp26 Hsp23 Hsp27 Hsp83 DnaJ-1 Hsf Jra kay Hr38 sr puc".split()
cells["stress"], _ = score(DISSOC)[0], None
cells["stress"] = score(DISSOC)[0]
for r in (MIT, ENDO):
    v = cells.stress[cells.role == r]
    print(f"Dissociation stress score, {r}: median {v.median():.2f}")
cut = min(np.quantile(cells.stress[cells.role == MIT], 0.75), np.quantile(cells.stress[cells.role == ENDO], 0.75))
cells["low_stress"] = cells.stress <= cut
print(f"Cells kept for the stress-matched comparison (score <= {cut:.2f}): "
      f"{int(((cells.role == MIT) & cells.low_stress).sum()):,} mitotic, "
      f"{int(((cells.role == ENDO) & cells.low_stress).sum()):,} endocycling")

In [ ]:
#@title Cell 6 - Redox programmes: endocycling vs mitotic follicle cells
PROGRAMMES = {
    "REDOX: glutathione S-transferases": "GstD1 GstD2 GstD3 GstD5 GstD9 GstE1 GstE3 GstE6 GstE7 GstE9 GstE12 GstE14 GstS1 GstO1 GstO2 GstO3 GstT1 GstZ1 GstZ2 Mgstl",
    "REDOX: glutathione synthesis & recycling": "Gclc Gclm Gss1 Gss2 Gr1 Grx1 Grx1t PHGPx GILT1",
    "REDOX: thioredoxin / peroxiredoxin": "Trxr-1 Trxr-2 Trx-2 TrxT dhd Jafrac1 Jafrac2 Prx3 Prx5 Prx2540-1 Prx2540-2 Srx",
    "REDOX: SOD and catalase": "Sod1 Sod2 Sod3 Cat",
    "REDOX: NADPH supply": "Zw Pgd Men Men-b Idh Idh3a Nmdmc Tkt Taldo",
    "REDOX: cnc (NRF2) targets": "cnc Keap1 GstD1 Gclc Gclm Cyp6a2 Cyp6a8 Ugt35a Jafrac1 Prx3",
    "REDOX: H2O2 / ROS-producing enzymes": "Ero1L Duox Nox Cyp4e3 Mav Uro Aox1 Pxd Pxt",
    "REDOX: mitochondrial respiration": "ND-75 ND-49 ND-30 ND-42 SdhA SdhB UQCR-C1 UQCR-C2 Cyt-c-p COX4 COX5A ATPsynbeta blw",
    "EXPLORE: ER folding load": "Xbp1 Hsc70-3 Ire1 PEK crc Pdi ERp60 Calr Ero1L Sec61alpha Sec61beta Srp54",
    "EXPLORE: ribosome / protein synthesis": "RpL3 RpL4 RpL5 RpL7 RpL8 RpL10 RpL11 RpL13 RpS2 RpS3 RpS4 RpS6 RpS8 RpS9 RpS11 RpS15",
    "EXPLORE: licensing and re-replication": "geminin dup Cdc6 Orc1 Mcm2 Mcm3 dpa Mcm5 Mcm6 Mcm7 CycE E2f1 Myc",
    "EXPLORE: p53 / DNA damage": "p53 Ku80 Irbp lok mei-41 tefu rad50 Xrp1 spn-A mus304",
    "CONTROL: heat shock": "Hsp70Aa Hsp83 Hsp68 Hsp27 Hsp26 Hsp23 DnaJ-1 Hsf",
    "CONTROL: immediate-early genes": "Hr38 sr kay Jra puc CrebA",
    "CONTROL: housekeeping": "Act5C alphaTub84B RpL32 Gapdh1 Tbp eIF4A Pgk Ef1alpha48D",
}
# Some genes are listed under a different name in the annotation, so each one is looked up under its synonyms too.
# FlyBase calls Geminin "geminin" (CG3183); "Gem" is only a synonym, which is why earlier runs could not find it.
ALIASES = {"geminin": ["Gem", "CG3183"], "dup": ["Cdt1"], "dpa": ["Mcm4"], "Zw": ["G6PD", "G6pd"],
           "Ero1L": ["Ero1", "CG1333"], "PHGPx": ["CG12013"], "Gss1": ["Gss"], "Gr1": ["Gr", "GR"],
           "Trxr-1": ["Trxr1"], "Jafrac1": ["Prx2"], "Men-b": ["Men-b", "CG5889"]}


def resolve(g, present):
    if g in present:
        return g
    for a in ALIASES.get(g, []):
        if a in present:
            return a
    return None


_renamed = {}
_resolved = {}
for n, v in PROGRAMMES.items():
    out = []
    for g in v.split():
        r = resolve(g, gpos)
        if r:
            out.append(r)
            if r != g:
                _renamed[g] = r
    _resolved[n] = out
PROGRAMMES = _resolved
if _renamed:
    print("Found under a synonym:", ", ".join(f"{k} -> {v}" for k, v in _renamed.items()))
_lic = [n for n in PROGRAMMES if "licensing" in n]
if _lic:
    print("Licensing set as tested:", ", ".join(PROGRAMMES[_lic[0]]))
for n, v in PROGRAMMES.items():
    print(f"{n}: {len(v)} genes found")
PROGRAMMES = {n: v for n, v in PROGRAMMES.items() if len(v) >= 3}

# Pseudobulk: add up the counts of each group, per library, then normalise by median of ratios.
# This avoids yolk protein genes (very high in endocycling cells) dominating the scale.
pb, lab = [], []
for lib in sorted(cells.library.unique()):
    for r in (MIT, ENDO):
        m = ((cells.library == lib) & (cells.role == r) & cells.low_stress).to_numpy()
        if m.sum() >= 50:
            pb.append(np.asarray(X[np.flatnonzero(m)].sum(axis=0)).ravel()); lab.append((lib, r))
pbd = pd.DataFrame(np.array(pb).T, index=genes, columns=pd.MultiIndex.from_tuples(lab))
pbd = pbd[(pbd / pbd.sum() * 1e6 >= 5).all(axis=1)]
lg = np.log(pbd.replace(0, np.nan)); ref = lg.mean(axis=1)
size = np.exp(lg[ref.notna()].sub(ref[ref.notna()], axis=0).median())
Lp = np.log2(pbd / size / pbd.sum().mean() * 1e6 + 1)
n_lib = len({l for l, _ in lab if (l, MIT) in lab and (l, ENDO) in lab})
print(f"\nStress-matched cells only. Pseudobulk groups: {len(lab)} ({n_lib} librar{'y' if n_lib == 1 else 'ies'} have both cell groups), "
      f"{len(Lp):,} genes expressed in all of them")
fc = Lp.xs(ENDO, level=1, axis=1).mean(axis=1) - Lp.xs(MIT, level=1, axis=1).mean(axis=1)
per_lib = {l: Lp[(l, ENDO)] - Lp[(l, MIT)] for l in sorted({a for a, b in lab}) if (l, ENDO) in lab and (l, MIT) in lab}

level = Lp.mean(axis=1)
bins = pd.qcut(level.rank(method="first"), 10, labels=False)
avoid = {g for v in PROGRAMMES.values() for g in v}
pools = {b: [g for g in bins.index[bins == b] if g not in avoid] for b in range(10)}
controls = [n for n in PROGRAMMES if n.startswith("CONTROL")]
rows = []
for n, gs in PROGRAMMES.items():
    gs = [g for g in gs if g in fc.index]
    if len(gs) < 3:
        continue
    obs = fc.loc[gs].median()
    null = np.array([fc.loc[[rng.choice(pools[bins[g]]) for g in gs]].median() for _ in range(N_RANDOM_SETS)])
    c = np.median(null)
    r = {"programme": n, "genes": len(gs), "log2_change": obs, "random_median": c,
         "p_vs_random": (1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (N_RANDOM_SETS + 1),
         "beyond_all_controls": all(np.sign(obs - fc.loc[[x for x in PROGRAMMES[k] if x in fc.index]].median())
                                    == np.sign(obs - c) for k in controls)}
    for l, d in per_lib.items():
        r[f"lib_{l[:14]}"] = d.loc[gs].median()
    rows.append(r)
res = pd.DataFrame(rows)
m = ~res.programme.str.startswith("CONTROL")
p = res.loc[m, "p_vs_random"].to_numpy(); o = np.argsort(p)
q = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
qq = np.empty_like(p); qq[o] = np.minimum(q, 1); res.loc[m, "q_fdr"] = qq
display(res.round(3))
print("log2_change: endocycling minus mitotic follicle cells. Positive means higher in endocycling cells.")
libcols = [c for c in res.columns if c.startswith("lib_")]
print("\nVerdict:")
for _, r in res[m].iterrows():
    agree = len(libcols) < 2 or len({np.sign(r[c] - res.loc[m, c].median()) for c in libcols}) == 1
    v = ("HIGHER" if r.log2_change > r.random_median else "LOWER") + " in endocycling follicle cells"
    if r.q_fdr >= 0.05:
        v = "no difference from random genes"
    elif not r.beyond_all_controls:
        v += " (but not beyond every stress/housekeeping control)"
    elif not agree:
        v += " (but the libraries disagree)"
    print(f"   {r.programme}: {v} ({r.log2_change:+.2f})")
if len(libcols) < 2:
    print("\nOnly one library has both cell groups, so there is no biological replication. The comparison with random "
          "gene sets is the only safeguard here: read the result as a lead, not as a measured effect size.")
print("\nNote: in the follicle cell lineage, ploidy and developmental stage rise together. A difference here means "
      "endocycling cells differ from mitotic ones, not that ploidy alone caused it.")

In [ ]:
#@title Cell 7 - Gene by gene, the graded stage trend, chart and saving
gl = list(dict.fromkeys(g for n, v in PROGRAMMES.items() if not n.startswith("CONTROL") for g in v if g in fc.index))
tab2 = pd.DataFrame({"log2_change": fc.loc[gl], "pct_of_all_genes": fc.rank(pct=True).loc[gl],
                     "mean_mitotic": Lp.xs(MIT, level=1, axis=1).mean(axis=1).loc[gl],
                     "mean_endocycling": Lp.xs(ENDO, level=1, axis=1).mean(axis=1).loc[gl]})
for l, d in per_lib.items():
    tab2[f"lib_{l[:14]}"] = d.loc[gl]
tab2["programme"] = [", ".join(n.split(": ")[1] for n, v in PROGRAMMES.items() if g in v) for g in gl]
display(tab2.sort_values("log2_change").round(2))

# Graded check: do the programmes change step by step across the follicle cell clusters, ordered by stage?
order = [cl for cl in summary.index if summary.loc[cl, "role"].startswith("follicle cell") or
         summary.loc[cl, "role"].endswith("follicle cell")]
order = sorted(order, key=lambda cl: stage[cl])
grade = {}
for n, gs in PROGRAMMES.items():
    s, _ = score(gs)
    grade[n.split(": ")[1]] = pd.Series(s).groupby(cells.cluster.to_numpy()).mean().reindex(order)
grade = pd.DataFrame(grade).T
grade.columns = [f"c{cl} ({summary.loc[cl, 'role'].replace(' follicle cell', '')}, stage {stage[cl]:+.1f})" for cl in order]
display(grade.round(2))
print("Follicle cell clusters left to right by stage score (earliest first). Values are mean log expression per cell.")

fig, ax = plt.subplots(figsize=(8, 0.34 * len(res) + 1.4))
col = ["#b2432f" if v < 0 else "#2f6db2" for v in (res.log2_change - res.random_median)]
ax.barh(res.programme, res.log2_change - res.random_median, color=col)
for i, r in res.reset_index().iterrows():
    ax.text(r.log2_change - r.random_median, i, f"  q={r.q_fdr:.3f}" if r.q_fdr == r.q_fdr else "  (control)",
            va="center", fontsize=8)
ax.axvline(0, color="black", lw=0.8); ax.invert_yaxis()
ax.set_xlabel("log2 change, endocycling vs mitotic follicle cells, minus random genes")
ax.set_title(f"{GSE}: programmes across the mitotic-to-endocycle switch", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(GDIR, "chart_programmes.png"), dpi=150); plt.show()
res.to_csv(os.path.join(GDIR, "programmes.csv"), index=False)
tab2.to_csv(os.path.join(GDIR, "genes.csv")); grade.to_csv(os.path.join(GDIR, "stage_trend.csv"))
cells[["library", "barcode", "cluster", "role", "n_counts", "n_genes"]].to_csv(os.path.join(GDIR, "cells.csv"), index=False)
print("Saved to", GDIR)